# Space–time cubes and cohort histories

**Time:** 35 minutes · **Prerequisites:** basic Python; run cells from top to bottom.

Construct a space–time cube from public aggregates; trace a city history; compare a cube with a heatmap.

[Run this notebook in JupyterLite](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/lite/lab/index.html?path=notebooks/08_space_time_cube.ipynb) · [Full-screen dashboard](https://jltobias.github.io/JupyterLite-Sea-Level-Rise/dashboard/) · [Official poster](https://plus.iasociety.org/sites/default/files/2024-09/e-poster_755.pdf)

Each notebook runs independently. Use **Run → Run All Cells**; the first package download can take a minute. Save a downloaded copy to keep your work. The read-only book includes executed examples.

In [1]:
# First run downloads Python packages in JupyterLite; subsequent runs use browser caches.
import sys
if sys.platform == 'emscripten':
    import piplite
    await piplite.install(['numpy', 'pandas', 'matplotlib', 'plotly', 'ipywidgets'])
from pathlib import Path
root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from IPython.display import display, HTML
from coastlab import (DATA, SCENARIOS, YEARS, COLORS, load_data, select, cards,
                      toy_slr, toy_probability, cumulative_probability,
                      haversine_km, show, dashboard, land_traces)
facilities, exposure = load_data()
published = pd.read_csv(DATA / 'poster_aggregates.csv')
print('Ready: public poster aggregates + clearly labeled fictional facility fixtures.')

Ready: public poster aggregates + clearly labeled fictional facility fixtures.


## Stack maps, not populations

Imagine each decade's map as a sheet of paper and stack the sheets in time order. Longitude and latitude locate a city; vertical position locates a year. Colour shows the published potentially flooded percentage. Repeated points are observations of the same reporting unit under different modeled years; they are not additional facilities.

Rotation can reveal patterns but also hide points. Offer a flat alternative with exact values. Keep the colour scale fixed across RCPs.

In [2]:
city=published.query("scope == 'city' and scenario == 'RCP 8.5'")
fig=go.Figure()
for name,g in city.groupby('city'):
    fig.add_trace(go.Scatter3d(x=g.longitude,y=g.latitude,z=g.year,mode='lines+markers',name=name,
        marker=dict(size=7,color=g.percent,cmin=0,cmax=100,colorscale='YlOrRd'),
        text=[f'{name}: {n} / {d}' for n,d in zip(g.potentially_flooded,g.facilities_2023)],
        hovertemplate='%{text}<br>Year %{z}<extra></extra>'))
fig.update_layout(title='Public city time histories — vertical axis = year',scene=dict(xaxis_title='Longitude',yaxis_title='Latitude',zaxis_title='Year'))
show(fig)

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="d9d39a0b-dc37-455f-8544-b4c4dc05b851" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("d9d39a0b-dc37-455f-8544-b4c4dc05b851")) { Plotly.newPlot( "d9d39a0b-dc37-455f-8544-b4c4dc05b851", [{"hovertemplate":"%{text}\u003cbr\u003eYear %{z}\u003cextra\u003e\u003c\u002fextra\u003e","marker":{"cmax":100,"cmin":0,"color":{"dtype":"f8","bdata":"K\u002faX3ZOHB0Ar9pfdk4cHQCv2l92ThwdAK\u002faX3ZOHB0CKsOHplZZUQIqw4emVllRAirDh6ZWWVECKsOHplZZUQA=="},"colorscale":[[0.0,"rgb(255,255,204)"],[0.125,"rgb(255,237,160)"],[0.25,"rgb(254,217,118)"],[0.375,"rgb(254,178,76)"],[0.5,"rgb(253,141,60)"],[0.625,"rgb(252,78,42)"],[0.75,"rgb(227,26,28)"],[0.875,"rgb(189,0,38)"],[1.0,"rgb(128,0,38)"]],"size":7},"mode":"lines+markers","name":"Abidjan","text":["Abidjan: 1 \u002f 34","Abidjan: 1 \u002f 34","Abidjan: 1 \u002f 34","Abidjan: 1 \u002f 34","Abidjan: 28 \u002f 34","Abidjan: 28 \u002f 34","Abidjan: 28 \u002f 34","Abidjan: 28 \u002f 34"],"x":{"dtype":"f8","bdata":"H4XrUbgeEMAfhetRuB4QwB+F61G4HhDAH4XrUbgeEMAfhetRuB4QwB+F61G4HhDAH4XrUbgeEMAfhetRuB4QwA=="},"y":{"dtype":"f8","bdata":"UrgehetRFUBSuB6F61EVQFK4HoXrURVAUrgehetRFUBSuB6F61EVQFK4HoXrURVAUrgehetRFUBSuB6F61EVQA=="},"z":{"dtype":"i2","bdata":"7gf4BwIIDAgWCCAIKgg0CA=="},"type":"scatter3d"},{"hovertemplate":"%{text}\u003cbr\u003eYear %{z}\u003cextra\u003e\u003c\u002fextra\u003e","marker":{"cmax":100,"cmin":0,"color":{"dtype":"f8","bdata":"HhZqTfMaUUAeFmpN8xpRQB4Wak3zGlFAFvvL7slrUkAW+8vuyWtSQBb7y+7Ja1JAvw6cM6K8U0C28\u002f3UeA1VQA=="},"colorscale":[[0.0,"rgb(255,255,204)"],[0.125,"rgb(255,237,160)"],[0.25,"rgb(254,217,118)"],[0.375,"rgb(254,178,76)"],[0.5,"rgb(253,141,60)"],[0.625,"rgb(252,78,42)"],[0.75,"rgb(227,26,28)"],[0.875,"rgb(189,0,38)"],[1.0,"rgb(128,0,38)"]],"size":7},"mode":"lines+markers","name":"Bangkok","text":["Bangkok: 13 \u002f 19","Bangkok: 13 \u002f 19","Bangkok: 13 \u002f 19","Bangkok: 14 \u002f 19","Bangkok: 14 \u002f 19","Bangkok: 14 \u002f 19","Bangkok: 15 \u002f 19","Bangkok: 16 \u002f 19"],"x":{"dtype":"f8","bdata":"AAAAAAAgWUAAAAAAACBZQAAAAAAAIFlAAAAAAAAgWUAAAAAAACBZQAAAAAAAIFlAAAAAAAAgWUAAAAAAACBZQA=="},"y":{"dtype":"f8","bdata":"AAAAAACAK0AAAAAAAIArQAAAAAAAgCtAAAAAAACAK0AAAAAAAIArQAAAAAAAgCtAAAAAAACAK0AAAAAAAIArQA=="},"z":{"dtype":"i2","bdata":"7gf4BwIIDAgWCCAIKgg0CA=="},"type":"scatter3d"},{"hovertemplate":"%{text}\u003cbr\u003eYear %{z}\u003cextra\u003e\u003c\u002fextra\u003e","marker":{"cmax":100,"cmin":0,"color":{"dtype":"f8","bdata":"AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAEtZhjjWRStAa5p3nKILQUBTliGOddFPQA=="},"colorscale":[[0.0,"rgb(255,255,204)"],[0.125,"rgb(255,237,160)"],[0.25,"rgb(254,217,118)"],[0.375,"rgb(254,178,76)"],[0.5,"rgb(253,141,60)"],[0.625,"rgb(252,78,42)"],[0.75,"rgb(227,26,28)"],[0.875,"rgb(189,0,38)"],[1.0,"rgb(128,0,38)"]],"size":7},"mode":"lines+markers","name":"Ho Chi Minh City","text":["Ho Chi Minh City: 0 \u002f 44","Ho Chi Minh City: 0 \u002f 44","Ho Chi Minh City: 0 \u002f 44","Ho Chi Minh City: 0 \u002f 44","Ho Chi Minh City: 0 \u002f 44","Ho Chi Minh City: 6 \u002f 44","Ho Chi Minh City: 15 \u002f 44","Ho Chi Minh City: 28 \u002f 44"],"x":{"dtype":"f8","bdata":"zczMzMysWkDNzMzMzKxaQM3MzMzMrFpAzczMzMysWkDNzMzMzKxaQM3MzMzMrFpAzczMzMysWkDNzMzMzKxaQA=="},"y":{"dtype":"f8","bdata":"j8L1KFyPJUCPwvUoXI8lQI\u002fC9ShcjyVAj8L1KFyPJUCPwvUoXI8lQI\u002fC9ShcjyVAj8L1KFyPJUCPwvUoXI8lQA=="},"z":{"dtype":"i2","bdata":"7gf4BwIIDAgWCCAIKgg0CA=="},"type":"scatter3d"},{"hovertemplate":"%{text}\u003cbr\u003eYear %{z}\u003cextra\u003e\u003c\u002fextra\u003e","marker":{"cmax":100,

In [3]:
matrix=city.pivot(index='city',columns='year',values='percent')
fig=px.imshow(matrix,zmin=0,zmax=100,text_auto='.1f',aspect='auto',color_continuous_scale='YlOrRd',
    labels={'color':'Potentially flooded (%)'},title='The same cube unfolded into a heatmap')
show(fig)

<iframe title="Interactive teaching figure" style="width:100%;height:560px;border:0" srcdoc="<!doctype html>
<html>
<head>
 <meta charset="utf-8" />
 <style>html, body {height: 100%;}</style>
</head>
<body>
 <div style="height:100%; width:100%;"> <script>window.PlotlyConfig = {MathJaxConfig: 'local'};</script>
 <script charset="utf-8" src="../../assets/plotly.min.js"></script> <div id="68826159-4295-4743-a532-1fa4462845fe" class="plotly-graph-div" style="height:100%; width:100%;"></div> <script> window.PLOTLYENV=window.PLOTLYENV || {}; if (document.getElementById("68826159-4295-4743-a532-1fa4462845fe")) { Plotly.newPlot( "68826159-4295-4743-a532-1fa4462845fe", [{"coloraxis":"coloraxis","name":"0","texttemplate":"%{z:.1f}","x":{"dtype":"i2","bdata":"7gf4BwIIDAgWCCAIKgg0CA=="},"y":["Abidjan","Bangkok","Ho Chi Minh City","Lagos","Mombasa"],"z":{"dtype":"f8","bdata":"K\u002faX3ZOHB0Ar9pfdk4cHQCv2l92ThwdAK\u002faX3ZOHB0CKsOHplZZUQIqw4emVllRAirDh6ZWWVECKsOHplZZUQB4Wak3zGlFAHhZqTfMaUUAeFmpN8xpRQBb7y+7Ja1JAFvvL7slrUkAW+8vuyWtSQL8OnDOivFNAtvP91HgNVUAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAS1mGONZFK0BrmnecogtBQFOWIY510U9AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAB6pSxDHOv1PwCRfvs6cBBAIbByaJEtI0DQRNjw9Oo1QAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA=","shape":"5, 8"},"type":"heatmap","xaxis":"x","yaxis":"y","hovertemplate":"year: %{x}\u003cbr\u003ecity: %{y}\u003cbr\u003ePotentially flooded (%): %{z}\u003cextra\u003e\u003c\u002fextra\u003e"}], {"template":{"data":{"histogram2dcontour":[{"type":"histogram2dcontour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"choropleth":[{"type":"choropleth","colorbar":{"outlinewidth":0,"ticks":""}}],"histogram2d":[{"type":"histogram2d","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"heatmap":[{"type":"heatmap","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"contourcarpet":[{"type":"contourcarpet","colorbar":{"outlinewidth":0,"ticks":""}}],"contour":[{"type":"contour","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"surface":[{"type":"surface","colorbar":{"outlinewidth":0,"ticks":""},"colorscale":[[0.0,"#0d0887"],[0.1111111111111111,"#46039f"],[0.2222222222222222,"#7201a8"],[0.3333333333333333,"#9c179e"],[0.4444444444444444,"#bd3786"],[0.5555555555555556,"#d8576b"],[0.6666666666666666,"#ed7953"],[0.7777777777777778,"#fb9f3a"],[0.8888888888888888,"#fdca26"],[1.0,"#f0f921"]]}],"mesh3d":[{"type":"mesh3d","colorbar":{"outlinewidth":0,"ticks":""}}],"scatter":[{"fillpattern":{"fillmode":"overlay","size":10,"solidity":0.2},"type":"scatter"}],"parcoords":[{"type":"parcoords","line":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"scatterpolargl":[{"type":"scatterpolargl","marker":{"colorbar":{"outlinewidth":0,"ticks":""}}}],"bar":[{"error_x":{"color":"#2a3f5f"},"error_y":{"color":"#2a3f5f"},"marker":{"line":{"color":"

## Lab: first crossing is not a predicted event date

Find the first sampled decade in which each city reaches 50% potentially flooded under each RCP. Return “not reached in sampled years” when appropriate. This is a threshold crossing in a modeled series, not the date of a real flood, and the ten-year sampling limits temporal precision.

In [4]:
city_rows=published.query("scope == 'city'")
crossings=city_rows[city_rows.percent>=50].groupby(['scenario','city']).year.min()
all_keys=pd.MultiIndex.from_product([SCENARIOS,sorted(city_rows.city.unique())],names=['scenario','city'])
display(crossings.reindex(all_keys).rename('first_sampled_year_at_50_percent').to_frame())

first_sampled_year_at_50_percent
scenario city                                              
RCP 2.6  Abidjan                                     2070.0
         Bangkok                                     2030.0
         Ho Chi Minh City                               NaN
         Lagos                                          NaN
         Mombasa                                        NaN
RCP 4.5  Abidjan                                     2070.0
         Bangkok                                     2030.0
         Ho Chi Minh City                               NaN
         Lagos                                          NaN
         Mombasa                                        NaN
RCP 8.5  Abidjan                                     2070.0
         Bangkok                                     2030.0
         Ho Chi Minh City                            2100.0
         Lagos                                          NaN
         Mombasa                                        NaN

**Worked check:** Bangkok's city row is already above 50% in 2030. Abidjan's city row first exceeds 50% in 2070. A missing value means the threshold is not reached in the available series; it should not become zero or 2020.

**Deliverable:** The cube, a heatmap and a one-sentence comparison of the questions each answers best. Avoid interpolating an exact transition date unless you state and defend an interpolation model.